# 第14章：强化学习交易 ⭐

## 本章学习目标

- 理解强化学习框架
- 掌握环境与模拟器设计
- 学会训练 RL 策略
- 能够评估执行效果

---

## 14.1 强化学习在交易中的应用

强化学习 (Reinforcement Learning, RL) 在量化交易中主要用于订单执行优化。

### 应用场景

```
订单执行问题:

目标: 在有限时间内完成大额订单，同时最小化交易成本

输入:                    输出:
- 订单方向 (买/卖)       - 每个时刻的下单量
- 订单总量               - 下单时机
- 时间限制               - 价格策略
- 市场状态               

RL 优势:
- 能学习复杂的市场动态
- 适应不同的市场环境
- 可以优化长期累积奖励
```

### 与传统方法对比

| 方法 | 特点 | 优点 | 缺点 |
|------|------|------|------|
| TWAP | 时间均匀分配 | 简单 | 不考虑市场状态 |
| VWAP | 成交量加权 | 成交量友好 | 需要成交量预测 |
| **RL** | 学习最优策略 | 自适应、最优 | 训练复杂 |

In [ ]:
import qlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 初始化 qlib
qlib.init(
    provider_uri="~/.qlib/qlib_data/cn_data",
    region="cn",
)

print("Qlib 初始化成功")

## 14.2 Qlib RL 框架概述

In [ ]:
# 查看 qlib RL 模块结构
from qlib import rl
import inspect

print("Qlib RL 模块:")
print("=" * 50)

# 列出 RL 相关模块
print("\n主要模块:")
print("  - qlib.rl.simulator: 模拟器")
print("  - qlib.rl.order_execution: 订单执行")
print("  - qlib.rl.trainer: 训练器")

## 14.3 RL 核心概念

In [ ]:
# RL 核心概念说明
print("强化学习核心概念:")
print("=" * 60)

concepts = {
    "Agent (智能体)": "执行交易决策的算法",
    "Environment (环境)": "市场模拟器，提供状态和奖励",
    "State (状态)": "当前市场信息和订单状态",
    "Action (动作)": "交易决策，如下单量",
    "Reward (奖励)": "执行效果的反馈信号",
    "Policy (策略)": "从状态到动作的映射",
}

for concept, desc in concepts.items():
    print(f"\n{concept}:")
    print(f"  {desc}")

In [ ]:
# 订单执行 RL 问题建模
print("\n订单执行 RL 问题建模:")
print("=" * 60)

print("\n状态空间 (State Space):")
print("  - 剩余订单量")
print("  - 剩余时间")
print("  - 当前价格")
print("  - 市场深度")
print("  - 成交量")
print("  - 波动率")

print("\n动作空间 (Action Space):")
print("  - 离散: 下单量 [0%, 25%, 50%, 75%, 100%]")
print("  - 连续: 下单量 [0, 1]")

print("\n奖励函数 (Reward Function):")
print("  - 执行成本: -成交价格偏离VWAP的程度")
print("  - 完成奖励: 订单完成时的额外奖励")
print("  - 惩罚项: 未完成订单的惩罚")

## 14.4 模拟器设计

In [ ]:
# 简化的订单执行模拟器
class SimpleExecutionSimulator:
    """简化的订单执行模拟器"""
    
    def __init__(self, 
                 total_amount=1000000,  # 订单总量
                 n_steps=10,           # 时间步数
                 volatility=0.02,      # 波动率
                 impact_coef=0.1):     # 冲击成本系数
        """
        参数:
            total_amount: 订单总量
            n_steps: 可执行的步数
            volatility: 价格波动率
            impact_coef: 市场冲击系数
        """
        self.total_amount = total_amount
        self.n_steps = n_steps
        self.volatility = volatility
        self.impact_coef = impact_coef
        
        self.reset()
    
    def reset(self):
        """重置环境"""
        self.current_step = 0
        self.remaining_amount = self.total_amount
        self.initial_price = 100.0  # 初始价格
        self.current_price = self.initial_price
        self.total_cost = 0
        self.executed_amount = 0
        
        return self._get_state()
    
    def _get_state(self):
        """获取当前状态"""
        return {
            'remaining_amount': self.remaining_amount / self.total_amount,
            'remaining_steps': (self.n_steps - self.current_step) / self.n_steps,
            'price_ratio': self.current_price / self.initial_price,
        }
    
    def step(self, action):
        """
        执行动作
        
        参数:
            action: 下单比例 [0, 1]
            
        返回:
            next_state: 下一状态
            reward: 奖励
            done: 是否结束
            info: 额外信息
        """
        # 计算下单量
        execute_amount = min(action * self.remaining_amount, self.remaining_amount)
        
        # 计算市场冲击
        impact = self.impact_coef * (execute_amount / self.total_amount)
        
        # 更新价格（随机波动 + 冲击）
        price_change = np.random.randn() * self.volatility + impact
        self.current_price *= (1 + price_change)
        
        # 计算成本
        cost = execute_amount * self.current_price * (1 + impact)
        self.total_cost += cost
        
        # 更新状态
        self.executed_amount += execute_amount
        self.remaining_amount -= execute_amount
        self.current_step += 1
        
        # 计算奖励 (负的成本)
        reward = -impact - abs(price_change) * 0.1
        
        # 检查是否结束
        done = (self.current_step >= self.n_steps) or (self.remaining_amount <= 0)
        
        # 最终奖励
        if done:
            # 未完成惩罚
            if self.remaining_amount > 0:
                reward -= 10 * (self.remaining_amount / self.total_amount)
            # 平均执行价格奖励
            avg_price = self.total_cost / self.executed_amount if self.executed_amount > 0 else self.initial_price
            reward -= (avg_price / self.initial_price - 1) * 100
        
        return self._get_state(), reward, done, {'executed': self.executed_amount}

# 测试模拟器
sim = SimpleExecutionSimulator()
state = sim.reset()

print("初始状态:", state)
print("\n执行测试动作...")

for i in range(5):
    action = 0.2  # 每次执行 20%
    state, reward, done, info = sim.step(action)
    print(f"Step {i+1}: state={state}, reward={reward:.4f}, done={done}")
    if done:
        break

## 14.5 策略网络

In [ ]:
# 简单的策略网络
import torch
import torch.nn as nn
import torch.optim as optim

class PolicyNetwork(nn.Module):
    """简单的策略网络"""
    
    def __init__(self, state_dim=3, hidden_dim=64, action_dim=1):
        super().__init__()
        
        self.network = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
            nn.Sigmoid(),  # 输出 [0, 1]
        )
    
    def forward(self, state):
        """前向传播"""
        if isinstance(state, dict):
            state = torch.tensor([
                state['remaining_amount'],
                state['remaining_steps'],
                state['price_ratio'],
            ], dtype=torch.float32)
        
        return self.network(state)

# 创建策略网络
policy = PolicyNetwork()

print("策略网络结构:")
print(policy)

## 14.6 训练循环

In [ ]:
# 简化的 REINFORCE 训练
def train_reinforce(env, policy, n_episodes=100, lr=0.01):
    """
    使用 REINFORCE 算法训练策略
    
    参数:
        env: 环境
        policy: 策略网络
        n_episodes: 训练轮数
        lr: 学习率
    """
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    
    episode_rewards = []
    
    for episode in range(n_episodes):
        # 收集轨迹
        log_probs = []
        rewards = []
        
        state = env.reset()
        
        while True:
            # 选择动作
            action_prob = policy(state)
            action = action_prob.item()
            
            # 执行动作
            next_state, reward, done, _ = env.step(action)
            
            # 存储轨迹
            log_prob = torch.log(action_prob + 1e-8)
            log_probs.append(log_prob)
            rewards.append(reward)
            
            state = next_state
            
            if done:
                break
        
        # 计算回报
        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + 0.99 * G
            returns.insert(0, G)
        
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        
        # 计算损失
        loss = 0
        for log_prob, G in zip(log_probs, returns):
            loss -= log_prob * G
        
        # 更新网络
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # 记录
        episode_rewards.append(sum(rewards))
        
        if (episode + 1) % 20 == 0:
            avg_reward = np.mean(episode_rewards[-20:])
            print(f"Episode {episode+1}: Avg Reward = {avg_reward:.4f}")
    
    return episode_rewards

# 训练
print("开始训练...")
rewards = train_reinforce(sim, policy, n_episodes=100)
print("\n训练完成")

In [ ]:
# 可视化训练过程
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(rewards, alpha=0.6)
plt.plot(pd.Series(rewards).rolling(10).mean(), linewidth=2, label='移动平均')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('训练奖励曲线')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(rewards, bins=20, edgecolor='black', alpha=0.7)
plt.xlabel('Total Reward')
plt.ylabel('Frequency')
plt.title('奖励分布')

plt.tight_layout()
plt.show()

## 14.7 与 TWAP 对比

In [ ]:
# TWAP 策略
def twap_strategy(env):
    """TWAP 策略：均匀分配订单"""
    state = env.reset()
    total_reward = 0
    
    twap_action = 1.0 / env.n_steps  # 每步执行相等的比例
    
    while True:
        _, reward, done, _ = env.step(twap_action)
        total_reward += reward
        if done:
            break
    
    return total_reward

# 对比 RL 和 TWAP
n_tests = 100

rl_rewards = []
twap_rewards = []

for _ in range(n_tests):
    # RL 策略
    state = sim.reset()
    rl_reward = 0
    while True:
        action = policy(state).item()
        _, reward, done, _ = sim.step(action)
        rl_reward += reward
        if done:
            break
    rl_rewards.append(rl_reward)
    
    # TWAP 策略
    twap_rewards.append(twap_strategy(sim))

# 对比结果
print("策略对比:")
print("=" * 40)
print(f"RL 策略: 平均奖励 = {np.mean(rl_rewards):.4f}, 标准差 = {np.std(rl_rewards):.4f}")
print(f"TWAP 策略: 平均奖励 = {np.mean(twap_rewards):.4f}, 标准差 = {np.std(twap_rewards):.4f}")

In [ ]:
# 可视化对比
plt.figure(figsize=(10, 5))

plt.hist(rl_rewards, bins=20, alpha=0.6, label='RL', edgecolor='black')
plt.hist(twap_rewards, bins=20, alpha=0.6, label='TWAP', edgecolor='black')
plt.axvline(x=np.mean(rl_rewards), color='blue', linestyle='--', label=f'RL Mean: {np.mean(rl_rewards):.2f}')
plt.axvline(x=np.mean(twap_rewards), color='orange', linestyle='--', label=f'TWAP Mean: {np.mean(twap_rewards):.2f}')

plt.xlabel('Total Reward')
plt.ylabel('Frequency')
plt.title('RL vs TWAP 奖励对比')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 14.8 使用 qlib RL 模块

In [ ]:
# qlib 内置 RL 功能说明
print("qlib RL 模块使用说明:")
print("=" * 60)

print("\n主要组件:")
print("  1. Simulator: 市场模拟器")
print("     - SimpleSimulator: 简化模拟")
print("     - QlibSimulator: 使用 qlib 数据")

print("\n  2. Policy: 策略网络")
print("     - 支持 PPO、SAC、DDPG 等")

print("\n  3. Trainer: 训练器")
print("     - 集成 tianshou 库")

print("\n注意: RL 模块需要 Linux 环境和额外的依赖")

In [ ]:
# 查看是否有 rl 相关示例
from pathlib import Path

examples_dir = Path("./examples")
if examples_dir.exists():
    rl_examples = list(examples_dir.rglob("*rl*.py"))
    if rl_examples:
        print("RL 示例文件:")
        for f in rl_examples[:5]:
            print(f"  - {f}")
    else:
        print("未找到 RL 示例文件")
else:
    print("examples 目录不存在")

## 14.9 实践练习

In [ ]:
# 练习1: 修改奖励函数
# 添加对未完成订单的惩罚

# 你的代码



# 提示：在 step 方法中修改奖励计算

In [ ]:
# 练习2: 实现不同的市场冲击模型
# 考虑成交量的平方根冲击

# 你的代码



# 提示：impact = coef * sqrt(volume / ADV)

In [ ]:
# 练习3: 实现一个简单的 DQN 算法
# 使用离散动作空间

# 你的代码



# 提示：动作空间 [0, 0.25, 0.5, 0.75, 1.0]

## 14.10 本章小结

本章我们学习了：

1. **强化学习在交易中的应用**：
   - 订单执行优化
   - 与 TWAP/VWAP 对比

2. **RL 核心概念**：
   - Agent, Environment, State, Action, Reward
   - 问题建模

3. **模拟器设计**：
   - 状态空间设计
   - 动作空间设计
   - 奖励函数设计

4. **策略训练**：
   - REINFORCE 算法
   - 策略网络
   - 训练循环

### 关键概念速查

| 概念 | 说明 |
|------|------|
| State | 剩余订单量、剩余时间、市场状态 |
| Action | 每步下单比例 |
| Reward | 负执行成本 + 完成奖励 |
| Policy | 状态到动作的映射 |

### 进一步学习

- PPO (Proximal Policy Optimization)
- SAC (Soft Actor-Critic)
- 多智能体 RL
- 实盘部署

### 下一章预告

下一章我们将学习在线服务与实盘部署，包括：
- 在线服务架构
- 实时数据更新
- 策略部署